# RAG Music Recommendation System - Complete Experiments

This notebook contains all experiments for the research paper:
- Multi-LLM comparison (Mistral-7B, LLaMA-2-7B, Gemma-9B)
- Comprehensive evaluation metrics
- Ablation studies
- Statistical significance tests
- Publication-ready tables and figures

**Authors**: Gautam Krishna M, Tanna Rakesh Naidu, Surige Sai Yashaswi, Subramaniyaswamy V

**Institution**: Vellore Institute of Technology

## Cell 1: Environment Setup & Installation

In [ ]:
# Install all required dependencies
!pip install -q accelerate bitsandbytes openpyxl seaborn textblob
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-cpu tiktoken sentence-transformers rank_bm25
!pip install -q transformers torch bert-score tqdm scipy
!pip install -q nltk beautifulsoup4 psutil tabulate

# Download NLTK data
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')

# Authenticate with Hugging Face
from huggingface_hub import login
login()  # Enter your HF token when prompted

## Cell 2: Imports & Configuration

In [ ]:
import os
import gc
import re
import json
import logging
import warnings
from datetime import datetime
from typing import List, Dict, Tuple, Optional, Any
from collections import defaultdict
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from tabulate import tabulate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder
from bert_score import score as bert_score

from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from bs4 import BeautifulSoup
from textblob import TextBlob
import psutil

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Cell 3: Configuration & LLM Models

In [ ]:
@dataclass
class Config:
    """Central configuration for experiments."""
    # Data
    DATA_PATH: str = "pitchfork_reviews.xlsx"
    VECTOR_STORE_PATH: str = "pitchfork_faiss_experiments"
    RESULTS_DIR: str = "experiment_results"
    
    # Data filtering
    MIN_YEAR: int = 2001
    MAX_YEAR: int = 2023
    MIN_REVIEW_LENGTH: int = 100
    
    # Chunking
    CHUNK_SIZE: int = 800
    CHUNK_OVERLAP: int = 100
    
    # Embedding
    EMBEDDING_MODEL: str = "sentence-transformers/all-mpnet-base-v2"
    EMBEDDING_BATCH_SIZE: int = 32
    
    # Retrieval
    BM25_K: int = 15
    VECTOR_K: int = 15
    DEFAULT_ENSEMBLE_WEIGHTS: List[float] = field(default_factory=lambda: [0.65, 0.35])
    
    # Reranking
    CROSS_ENCODER_MODEL: str = "cross-encoder/ms-marco-MiniLM-L-12-v2"
    RERANK_TOP_N: int = 5
    SCORE_BOOST_HIGH: float = 0.1
    SCORE_BOOST_MEDIUM: float = 0.05
    BEST_NEW_BOOST: float = 0.15
    TEMPORAL_BOOST: float = 0.08
    
    # LLM
    MAX_NEW_TOKENS: int = 500
    TEMPERATURE: float = 0.3
    TOP_P: float = 0.9
    
    # Evaluation
    EVAL_K_VALUES: List[int] = field(default_factory=lambda: [1, 3, 5, 10])
    BOOTSTRAP_SAMPLES: int = 1000
    CONFIDENCE_LEVEL: float = 0.95

config = Config()

# Create results directory
os.makedirs(config.RESULTS_DIR, exist_ok=True)

# LLM configurations for comparison
LLM_CONFIGS = {
    "Mistral-7B": {
        "model_id": "mistralai/Mistral-7B-Instruct-v0.2",
        "description": "Mistral AI's instruction-tuned 7B model"
    },
    "LLaMA-2-7B": {
        "model_id": "meta-llama/Llama-2-7b-chat-hf",
        "description": "Meta's LLaMA 2 chat model"
    },
    "Gemma-7B": {
        "model_id": "google/gemma-7b-it",
        "description": "Google's Gemma instruction-tuned model"
    }
}

# Genre hierarchy for evaluation
GENRE_HIERARCHY = {
    "rock": ["alternative", "indie rock", "punk", "post-punk", "shoegaze", "grunge"],
    "electronic": ["ambient", "techno", "house", "idm", "synth-pop", "downtempo", "trip-hop"],
    "hip-hop": ["rap", "trap", "boom bap", "drill", "grime"],
    "pop": ["synth-pop", "art pop", "indie pop", "dream pop", "electropop"],
    "jazz": ["fusion", "free jazz", "bebop", "acid jazz"],
    "folk": ["indie folk", "singer-songwriter", "americana"],
    "metal": ["black metal", "death metal", "doom metal", "progressive metal"],
    "r&b": ["neo-soul", "funk", "soul"]
}

# Query expansion synonyms
MUSIC_QUERY_EXPANSIONS = {
    "chill": ["calm", "relaxing", "mellow", "peaceful", "ambient"],
    "upbeat": ["energetic", "lively", "danceable", "uptempo"],
    "sad": ["melancholic", "emotional", "somber", "introspective"],
    "dreamy": ["ethereal", "hazy", "shoegaze", "ambient", "floating"],
    "heavy": ["intense", "loud", "aggressive", "powerful"]
}

print("Configuration loaded successfully!")
print(f"LLMs to compare: {list(LLM_CONFIGS.keys())}")

## Cell 4: Data Loading & Processing

In [ ]:
def load_and_clean_data(filepath: str) -> pd.DataFrame:
    """Load and clean the Pitchfork reviews dataset."""
    logger.info(f"Loading data from {filepath}")
    
    df = pd.read_excel(filepath, sheet_name="Result 1")
    initial_count = len(df)
    
    # Replace "Not Available" with NaN
    df = df.replace("Not Available", np.nan)
    
    # Clean HTML
    def clean_html(text):
        if pd.isna(text):
            return ""
        soup = BeautifulSoup(str(text), "html.parser")
        return re.sub(r'\s+', ' ', soup.get_text(separator=" ")).strip()
    
    df['review'] = df['review'].apply(clean_html)
    df['summary'] = df['summary'].apply(clean_html)
    
    # Convert types
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    df['year'] = pd.to_numeric(df['year'], errors='coerce')
    
    # Filter
    df = df[(df['year'] >= config.MIN_YEAR) & (df['year'] <= config.MAX_YEAR)]
    df['best_new'] = df['best_new'].apply(lambda x: str(x).lower() in ['true', '1', 'yes'])
    
    # Drop missing
    df = df.dropna(subset=['artist', 'album', 'score', 'year', 'review'])
    df = df[df['review'].str.len() >= config.MIN_REVIEW_LENGTH]
    df['genre'] = df['genre'].fillna('Unknown')
    
    # Remove duplicates
    df = df.drop_duplicates(subset=['artist', 'album'], keep='first').reset_index(drop=True)
    
    logger.info(f"Loaded {len(df)} reviews (from {initial_count})")
    return df

def get_score_category(score: float) -> str:
    if score >= 9.0: return "exceptional"
    elif score >= 8.0: return "excellent"
    elif score >= 7.0: return "good"
    elif score >= 6.0: return "decent"
    else: return "poor"

def get_decade(year: int) -> str:
    return f"{(int(year) // 10) * 10}s"

def create_documents(df: pd.DataFrame) -> List[Document]:
    """Convert DataFrame to LangChain Documents."""
    documents = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Creating documents"):
        content = f"Summary: {row['summary']}\n\nReview: {row['review']}"
        metadata = {
            'artist': str(row['artist']),
            'album': str(row['album']),
            'score': float(row['score']),
            'year': int(row['year']),
            'genre': str(row['genre']),
            'best_new': bool(row['best_new']),
            'score_category': get_score_category(row['score']),
            'decade': get_decade(row['year']),
            'artist_album': f"{row['artist']} - {row['album']}"
        }
        documents.append(Document(page_content=content, metadata=metadata))
    return documents

def chunk_documents(documents: List[Document]) -> List[Document]:
    """Chunk documents for RAG."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", ", ", " "]
    )
    chunked = []
    for doc in tqdm(documents, desc="Chunking"):
        chunks = splitter.split_text(doc.page_content)
        for i, chunk in enumerate(chunks):
            meta = doc.metadata.copy()
            meta['chunk_index'] = i
            chunked.append(Document(page_content=chunk, metadata=meta))
    return chunked

# Load and process data
df = load_and_clean_data(config.DATA_PATH)
documents = create_documents(df)
chunked_docs = chunk_documents(documents)

print(f"\nDataset Statistics:")
print(f"  Total reviews: {len(df)}")
print(f"  Unique artists: {df['artist'].nunique()}")
print(f"  Unique albums: {df['album'].nunique()}")
print(f"  Total chunks: {len(chunked_docs)}")
print(f"  Year range: {int(df['year'].min())}-{int(df['year'].max())}")
print(f"  Avg score: {df['score'].mean():.2f}")

## Cell 5: Vector Store & Retriever Setup

In [ ]:
def setup_embeddings():
    """Initialize embedding model."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    return HuggingFaceEmbeddings(
        model_name=config.EMBEDDING_MODEL,
        model_kwargs={'device': device},
        encode_kwargs={'normalize_embeddings': True, 'batch_size': config.EMBEDDING_BATCH_SIZE}
    )

def create_or_load_vector_store(documents, embeddings):
    """Create or load FAISS vector store."""
    if os.path.exists(config.VECTOR_STORE_PATH):
        logger.info(f"Loading vector store from {config.VECTOR_STORE_PATH}")
        return FAISS.load_local(config.VECTOR_STORE_PATH, embeddings, allow_dangerous_deserialization=True)
    
    logger.info("Creating new vector store...")
    batch_size = 1000
    vector_store = None
    
    for i in tqdm(range(0, len(documents), batch_size), desc="Building index"):
        batch = documents[i:i + batch_size]
        if vector_store is None:
            vector_store = FAISS.from_documents(batch, embeddings)
        else:
            vector_store.merge_from(FAISS.from_documents(batch, embeddings))
        if torch.cuda.is_available() and i % 5000 == 0:
            torch.cuda.empty_cache()
    
    vector_store.save_local(config.VECTOR_STORE_PATH)
    return vector_store

def setup_retrievers(vector_store, documents):
    """Set up all retriever types."""
    retrievers = {
        'vector': vector_store.as_retriever(search_type="similarity", search_kwargs={"k": config.VECTOR_K}),
        'bm25': BM25Retriever.from_documents(documents)
    }
    retrievers['bm25'].k = config.BM25_K
    retrievers['ensemble'] = EnsembleRetriever(
        retrievers=[retrievers['vector'], retrievers['bm25']],
        weights=config.DEFAULT_ENSEMBLE_WEIGHTS
    )
    return retrievers

# Setup
print("Setting up embeddings and vector store...")
embeddings = setup_embeddings()
vector_store = create_or_load_vector_store(chunked_docs, embeddings)
retrievers = setup_retrievers(vector_store, chunked_docs)
cross_encoder = CrossEncoder(config.CROSS_ENCODER_MODEL, max_length=512, 
                             device='cuda' if torch.cuda.is_available() else 'cpu')

print("\nRetrieval system ready!")
print(f"  Vector store: {vector_store.index.ntotal} vectors")
print(f"  Retrievers: {list(retrievers.keys())}")

## Cell 6: LLM Loader & RAG Chain Functions

In [ ]:
def load_llm(model_name: str) -> HuggingFacePipeline:
    """Load an LLM with 4-bit quantization."""
    model_id = LLM_CONFIGS[model_name]["model_id"]
    logger.info(f"Loading {model_name} ({model_id})...")
    
    # Clear GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
    
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )
    
    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=config.MAX_NEW_TOKENS,
        temperature=config.TEMPERATURE,
        top_p=config.TOP_P,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
    
    return HuggingFacePipeline(pipeline=pipe), tokenizer

def unload_llm():
    """Unload LLM to free GPU memory."""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

# Query processing functions
def expand_query(query: str) -> str:
    """Expand query with synonyms."""
    expanded = [query]
    query_lower = query.lower()
    
    for term, synonyms in MUSIC_QUERY_EXPANSIONS.items():
        if term in query_lower:
            expanded.extend(synonyms[:3])
    
    for genre, subgenres in GENRE_HIERARCHY.items():
        if genre in query_lower:
            expanded.extend(subgenres[:3])
    
    return " ".join(list(dict.fromkeys(expanded)))

def detect_intent(query: str) -> Dict[str, Any]:
    """Detect query intent."""
    query_lower = query.lower()
    intent = {
        'is_artist_specific': any(w[0].isupper() for w in query.split() if len(w) > 2),
        'is_temporal': any(t in query_lower for t in ['recent', 'latest', 'new', 'old', 'classic']),
        'is_mood_based': any(t in query_lower for t in MUSIC_QUERY_EXPANSIONS.keys()),
        'prefers_recent': any(t in query_lower for t in ['recent', 'latest', 'new']),
        'year_mentioned': None
    }
    year_match = re.search(r'\b(19|20)\d{2}\b', query)
    if year_match:
        intent['year_mentioned'] = int(year_match.group())
    return intent

def get_adaptive_weights(intent: Dict) -> List[float]:
    """Get adaptive ensemble weights."""
    v, b = config.DEFAULT_ENSEMBLE_WEIGHTS
    if intent['is_artist_specific']: b += 0.15; v -= 0.15
    if intent['is_mood_based']: v += 0.1; b -= 0.1
    total = v + b
    return [v/total, b/total]

def rerank_documents(query: str, docs: List[Document], intent: Dict) -> List[Document]:
    """Rerank with cross-encoder and metadata boosts."""
    if not docs:
        return []
    
    # Deduplicate
    seen = {}
    for doc in docs:
        key = doc.metadata.get('artist_album')
        if key not in seen or len(doc.page_content) > len(seen[key].page_content):
            seen[key] = doc
    docs = list(seen.values())
    
    # Score
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    
    # Boost
    boosted = []
    for doc, score in zip(docs, scores):
        boost = 0
        if doc.metadata.get('score', 0) >= 8.0: boost += config.SCORE_BOOST_HIGH
        elif doc.metadata.get('score', 0) >= 7.0: boost += config.SCORE_BOOST_MEDIUM
        if doc.metadata.get('best_new'): boost += config.BEST_NEW_BOOST
        if intent.get('prefers_recent') and doc.metadata.get('year', 0) >= 2020:
            boost += config.TEMPORAL_BOOST
        if intent.get('year_mentioned') == doc.metadata.get('year'):
            boost += 0.2
        boosted.append((doc, score + boost))
    
    boosted.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in boosted[:config.RERANK_TOP_N]]

def retrieve_documents(query: str, use_expansion=True, use_adaptive=True, use_reranking=True) -> List[Document]:
    """Full retrieval pipeline."""
    intent = detect_intent(query)
    search_query = expand_query(query) if use_expansion else query
    
    if use_adaptive:
        weights = get_adaptive_weights(intent)
        ensemble = EnsembleRetriever(
            retrievers=[retrievers['vector'], retrievers['bm25']],
            weights=weights
        )
        docs = ensemble.invoke(search_query)
    else:
        docs = retrievers['ensemble'].invoke(search_query)
    
    if use_reranking:
        docs = rerank_documents(query, docs, intent)
    else:
        docs = docs[:config.RERANK_TOP_N]
    
    return docs

print("LLM loader and retrieval functions ready!")

## Cell 7: Test Queries with Ground Truth

In [ ]:
# Comprehensive test queries with attribute-based ground truth
TEST_QUERIES = [
    # Genre-specific queries (30)
    {"query": "dreamy shoegaze albums", "type": "genre", "expected_genres": ["rock", "electronic"], "expected_keywords": ["shoegaze", "dream"]},
    {"query": "ambient electronic music", "type": "genre", "expected_genres": ["electronic"], "expected_keywords": ["ambient"]},
    {"query": "aggressive hip-hop with hard beats", "type": "genre", "expected_genres": ["rap", "hip-hop"]},
    {"query": "smooth jazz fusion albums", "type": "genre", "expected_genres": ["jazz"]},
    {"query": "indie folk singer-songwriter", "type": "genre", "expected_genres": ["folk", "rock"]},
    {"query": "post-punk revival bands", "type": "genre", "expected_genres": ["rock"]},
    {"query": "experimental noise rock", "type": "genre", "expected_genres": ["rock", "experimental"]},
    {"query": "neo-soul r&b albums", "type": "genre", "expected_genres": ["r&b", "pop"]},
    {"query": "psychedelic rock from the 2010s", "type": "genre", "expected_genres": ["rock"], "year_range": [2010, 2019]},
    {"query": "trap music with melodic hooks", "type": "genre", "expected_genres": ["rap", "hip-hop"]},
    {"query": "progressive metal albums", "type": "genre", "expected_genres": ["metal", "rock"]},
    {"query": "synth-pop and new wave", "type": "genre", "expected_genres": ["electronic", "pop", "rock"]},
    {"query": "doom metal slow and heavy", "type": "genre", "expected_genres": ["metal", "rock"]},
    {"query": "trip-hop downtempo", "type": "genre", "expected_genres": ["electronic"]},
    {"query": "garage rock raw energy", "type": "genre", "expected_genres": ["rock"]},
    {"query": "chamber pop orchestral", "type": "genre", "expected_genres": ["rock", "pop"]},
    {"query": "IDM glitchy electronic", "type": "genre", "expected_genres": ["electronic"]},
    {"query": "emo and screamo albums", "type": "genre", "expected_genres": ["rock"]},
    {"query": "afrobeat and world music", "type": "genre", "expected_genres": ["global", "rock", "pop"]},
    {"query": "dream pop ethereal vocals", "type": "genre", "expected_genres": ["rock", "pop", "electronic"]},
    {"query": "hardcore punk fast and loud", "type": "genre", "expected_genres": ["rock"]},
    {"query": "minimalist classical compositions", "type": "genre", "expected_genres": ["experimental"]},
    {"query": "funk and disco grooves", "type": "genre", "expected_genres": ["pop", "r&b"]},
    {"query": "lo-fi bedroom pop", "type": "genre", "expected_genres": ["rock", "pop"]},
    {"query": "industrial and EBM", "type": "genre", "expected_genres": ["electronic", "rock"]},
    {"query": "bluegrass and country folk", "type": "genre", "expected_genres": ["folk", "country"]},
    {"query": "grime UK rap", "type": "genre", "expected_genres": ["rap", "electronic"]},
    {"query": "math rock complex rhythms", "type": "genre", "expected_genres": ["rock"]},
    {"query": "black metal atmospheric", "type": "genre", "expected_genres": ["metal"]},
    {"query": "vaporwave and chillwave", "type": "genre", "expected_genres": ["electronic"]},
    
    # Temporal queries (20)
    {"query": "best albums from 2019", "type": "temporal", "year_range": [2019, 2019], "min_score": 8.0},
    {"query": "2020 hip-hop releases", "type": "temporal", "expected_genres": ["rap", "hip-hop"], "year_range": [2020, 2020]},
    {"query": "recent indie rock 2021 2022", "type": "temporal", "expected_genres": ["rock"], "year_range": [2021, 2022]},
    {"query": "classic 2000s alternative", "type": "temporal", "expected_genres": ["rock"], "year_range": [2000, 2009]},
    {"query": "early 2010s electronic", "type": "temporal", "expected_genres": ["electronic"], "year_range": [2010, 2013]},
    {"query": "2018 jazz albums", "type": "temporal", "expected_genres": ["jazz"], "year_range": [2018, 2018]},
    {"query": "late 2010s pop music", "type": "temporal", "expected_genres": ["pop"], "year_range": [2017, 2019]},
    {"query": "2015 metal releases", "type": "temporal", "expected_genres": ["metal", "rock"], "year_range": [2015, 2015]},
    {"query": "mid 2000s indie", "type": "temporal", "expected_genres": ["rock"], "year_range": [2004, 2007]},
    {"query": "2023 album releases", "type": "temporal", "year_range": [2023, 2023]},
    {"query": "2017 r&b and soul", "type": "temporal", "expected_genres": ["r&b", "pop"], "year_range": [2017, 2017]},
    {"query": "2012 folk albums", "type": "temporal", "expected_genres": ["folk"], "year_range": [2012, 2012]},
    {"query": "best of 2016 all genres", "type": "temporal", "year_range": [2016, 2016], "min_score": 8.0},
    {"query": "2014 experimental music", "type": "temporal", "expected_genres": ["experimental", "electronic"], "year_range": [2014, 2014]},
    {"query": "recent rap 2022", "type": "temporal", "expected_genres": ["rap"], "year_range": [2022, 2022]},
    {"query": "2011 electronic albums", "type": "temporal", "expected_genres": ["electronic"], "year_range": [2011, 2011]},
    {"query": "2008 indie rock gems", "type": "temporal", "expected_genres": ["rock"], "year_range": [2008, 2008]},
    {"query": "latest 2023 pop releases", "type": "temporal", "expected_genres": ["pop"], "year_range": [2023, 2023]},
    {"query": "2013 hip-hop classics", "type": "temporal", "expected_genres": ["rap"], "year_range": [2013, 2013]},
    {"query": "2010 breakthrough albums", "type": "temporal", "year_range": [2010, 2010], "min_score": 8.5},
    
    # Mood-based queries (25)
    {"query": "chill relaxing music for studying", "type": "mood", "expected_genres": ["electronic", "jazz", "folk"]},
    {"query": "upbeat energetic workout music", "type": "mood", "expected_genres": ["electronic", "rock", "rap"]},
    {"query": "sad melancholic albums for rainy days", "type": "mood", "expected_genres": ["folk", "rock", "pop"]},
    {"query": "happy uplifting feel-good music", "type": "mood", "expected_genres": ["pop", "rock"]},
    {"query": "dark brooding atmospheric albums", "type": "mood", "expected_genres": ["electronic", "metal", "rock"]},
    {"query": "peaceful calming ambient", "type": "mood", "expected_genres": ["electronic"]},
    {"query": "intense aggressive music", "type": "mood", "expected_genres": ["metal", "rock", "rap"]},
    {"query": "romantic love songs", "type": "mood", "expected_genres": ["pop", "r&b", "folk"]},
    {"query": "nostalgic retro vibes", "type": "mood", "expected_genres": ["pop", "electronic", "rock"]},
    {"query": "meditative introspective albums", "type": "mood", "expected_genres": ["electronic", "folk", "jazz"]},
    {"query": "party dance music", "type": "mood", "expected_genres": ["electronic", "pop"]},
    {"query": "mellow evening listening", "type": "mood", "expected_genres": ["jazz", "folk", "electronic"]},
    {"query": "epic cinematic soundscapes", "type": "mood", "expected_genres": ["electronic", "rock"]},
    {"query": "gloomy depressive music", "type": "mood", "expected_genres": ["metal", "rock", "folk"]},
    {"query": "triumphant victorious anthems", "type": "mood", "expected_genres": ["rock", "metal"]},
    {"query": "anxious tense music", "type": "mood", "expected_genres": ["electronic", "rock"]},
    {"query": "warm comforting albums", "type": "mood", "expected_genres": ["folk", "pop", "jazz"]},
    {"query": "rebellious punk attitude", "type": "mood", "expected_genres": ["rock"]},
    {"query": "mystical spiritual music", "type": "mood", "expected_genres": ["electronic", "folk", "global"]},
    {"query": "playful fun quirky albums", "type": "mood", "expected_genres": ["pop", "rock"]},
    {"query": "soothing sleep music", "type": "mood", "expected_genres": ["electronic", "jazz"]},
    {"query": "raw emotional cathartic", "type": "mood", "expected_genres": ["rock", "folk", "rap"]},
    {"query": "summer beach vibes", "type": "mood", "expected_genres": ["pop", "electronic", "rock"]},
    {"query": "winter cold atmospheric", "type": "mood", "expected_genres": ["electronic", "metal", "folk"]},
    {"query": "morning coffee acoustic", "type": "mood", "expected_genres": ["folk", "jazz", "pop"]},
    
    # Quality-based queries (15)
    {"query": "highest rated albums of all time", "type": "quality", "min_score": 9.0},
    {"query": "best new music pitchfork picks", "type": "quality", "best_new": True, "min_score": 8.0},
    {"query": "critically acclaimed hip-hop", "type": "quality", "expected_genres": ["rap"], "min_score": 8.5},
    {"query": "perfect 10 albums", "type": "quality", "min_score": 9.5},
    {"query": "underrated gems low scores", "type": "quality", "max_score": 7.0},
    {"query": "essential rock albums highly rated", "type": "quality", "expected_genres": ["rock"], "min_score": 8.5},
    {"query": "best electronic albums ever", "type": "quality", "expected_genres": ["electronic"], "min_score": 8.5},
    {"query": "acclaimed debut albums", "type": "quality", "min_score": 8.0},
    {"query": "top rated jazz records", "type": "quality", "expected_genres": ["jazz"], "min_score": 8.0},
    {"query": "masterpiece folk albums", "type": "quality", "expected_genres": ["folk"], "min_score": 8.5},
    {"query": "legendary metal albums", "type": "quality", "expected_genres": ["metal"], "min_score": 8.0},
    {"query": "exceptional pop records", "type": "quality", "expected_genres": ["pop"], "min_score": 8.5},
    {"query": "groundbreaking experimental", "type": "quality", "expected_genres": ["experimental", "electronic"], "min_score": 8.0},
    {"query": "best reviewed rap albums", "type": "quality", "expected_genres": ["rap"], "min_score": 8.5},
    {"query": "award worthy albums", "type": "quality", "min_score": 8.5, "best_new": True},
    
    # Complex multi-attribute queries (10)
    {"query": "highly rated recent shoegaze 2020s", "type": "complex", "expected_genres": ["rock"], "year_range": [2020, 2023], "min_score": 7.5},
    {"query": "best new music hip-hop 2019", "type": "complex", "expected_genres": ["rap"], "year_range": [2019, 2019], "best_new": True},
    {"query": "chill electronic from 2015 or later", "type": "complex", "expected_genres": ["electronic"], "year_range": [2015, 2023]},
    {"query": "acclaimed indie folk 2010s melancholic", "type": "complex", "expected_genres": ["folk", "rock"], "year_range": [2010, 2019], "min_score": 8.0},
    {"query": "energetic punk rock 2000s", "type": "complex", "expected_genres": ["rock"], "year_range": [2000, 2009]},
    {"query": "dreamy pop highly rated 2018", "type": "complex", "expected_genres": ["pop", "rock"], "year_range": [2018, 2018], "min_score": 8.0},
    {"query": "dark ambient electronic 2010s", "type": "complex", "expected_genres": ["electronic"], "year_range": [2010, 2019]},
    {"query": "acclaimed jazz fusion recent", "type": "complex", "expected_genres": ["jazz"], "year_range": [2018, 2023], "min_score": 7.5},
    {"query": "best metal albums 2015-2020", "type": "complex", "expected_genres": ["metal"], "year_range": [2015, 2020], "min_score": 8.0},
    {"query": "uplifting pop 2021 best new", "type": "complex", "expected_genres": ["pop"], "year_range": [2021, 2021], "best_new": True}
]

print(f"Total test queries: {len(TEST_QUERIES)}")
print(f"Query types: {dict(pd.Series([q['type'] for q in TEST_QUERIES]).value_counts())}")

## Cell 8: Evaluation Metrics

In [ ]:
def calculate_mrr(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Mean Reciprocal Rank at K."""
    for i, item in enumerate(retrieved[:k]):
        if item.lower() in [r.lower() for r in relevant]:
            return 1.0 / (i + 1)
    return 0.0

def calculate_precision(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Precision at K."""
    if k == 0: return 0.0
    hits = sum(1 for r in retrieved[:k] if r.lower() in [rel.lower() for rel in relevant])
    return hits / k

def calculate_recall(retrieved: List[str], relevant: List[str], k: int) -> float:
    """Recall at K."""
    if not relevant: return 0.0
    hits = sum(1 for r in retrieved[:k] if r.lower() in [rel.lower() for rel in relevant])
    return hits / len(relevant)

def calculate_ndcg(retrieved: List[str], relevant: List[str], k: int) -> float:
    """NDCG at K."""
    dcg = sum(1.0 / np.log2(i + 2) for i, item in enumerate(retrieved[:k]) 
              if item.lower() in [r.lower() for r in relevant])
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def calculate_genre_diversity(docs: List[Document]) -> float:
    """Genre diversity using entropy."""
    if not docs: return 0.0
    genres = [doc.metadata.get('genre', 'Unknown') for doc in docs]
    counts = defaultdict(int)
    for g in genres: counts[g] += 1
    total = len(genres)
    entropy = -sum((c/total) * np.log2(c/total) for c in counts.values() if c > 0)
    max_entropy = np.log2(len(genres)) if len(genres) > 1 else 1
    return entropy / max_entropy if max_entropy > 0 else 0.0

def evaluate_attributes(docs: List[Document], expected: Dict) -> Dict[str, float]:
    """Evaluate attribute matching."""
    if not docs:
        return {'genre_match': 0, 'year_match': 0, 'score_match': 0}
    
    results = {'genre_match': 0, 'year_match': 0, 'score_match': 0}
    n = len(docs)
    
    for doc in docs:
        # Genre match
        if 'expected_genres' in expected:
            doc_genre = doc.metadata.get('genre', '').lower()
            if any(g in doc_genre for g in expected['expected_genres']):
                results['genre_match'] += 1
        else:
            results['genre_match'] += 1  # No constraint
        
        # Year match
        if 'year_range' in expected:
            year = doc.metadata.get('year', 0)
            if expected['year_range'][0] <= year <= expected['year_range'][1]:
                results['year_match'] += 1
        else:
            results['year_match'] += 1
        
        # Score match
        score = doc.metadata.get('score', 0)
        min_score = expected.get('min_score', 0)
        max_score = expected.get('max_score', 10)
        if min_score <= score <= max_score:
            results['score_match'] += 1
        
        # Best new match
        if expected.get('best_new'):
            if doc.metadata.get('best_new'):
                pass  # Already counted in score
    
    return {k: v/n for k, v in results.items()}

def bootstrap_confidence_interval(data: List[float], confidence: float = 0.95, n_samples: int = 1000) -> Tuple[float, float]:
    """Calculate bootstrap confidence interval."""
    if not data:
        return (0.0, 0.0)
    bootstrapped = []
    for _ in range(n_samples):
        sample = np.random.choice(data, size=len(data), replace=True)
        bootstrapped.append(np.mean(sample))
    lower = np.percentile(bootstrapped, (1 - confidence) / 2 * 100)
    upper = np.percentile(bootstrapped, (1 + confidence) / 2 * 100)
    return (lower, upper)

print("Evaluation metrics ready!")

## Cell 9: BERTScore & Faithfulness Evaluation

In [ ]:
def calculate_bertscore(generated: List[str], references: List[str]) -> Dict[str, float]:
    """Calculate BERTScore for generated text."""
    if not generated or not references:
        return {'precision': 0, 'recall': 0, 'f1': 0}
    
    P, R, F1 = bert_score(generated, references, lang='en', verbose=False)
    return {
        'precision': P.mean().item(),
        'recall': R.mean().item(),
        'f1': F1.mean().item()
    }

def calculate_faithfulness(generated: str, context_docs: List[Document]) -> float:
    """Calculate faithfulness score (claims supported by context)."""
    if not generated or not context_docs:
        return 0.0
    
    # Extract mentioned albums/artists from generated text
    generated_lower = generated.lower()
    context_items = set()
    
    for doc in context_docs:
        context_items.add(doc.metadata['artist'].lower())
        context_items.add(doc.metadata['album'].lower())
    
    # Check how many context items are mentioned
    mentioned = sum(1 for item in context_items if item in generated_lower)
    
    return mentioned / len(context_items) if context_items else 0.0

def evaluate_generation_quality(query: str, generated: str, context_docs: List[Document]) -> Dict[str, float]:
    """Comprehensive generation quality evaluation."""
    # Create reference from context
    reference = " ".join([f"{d.metadata['artist']} {d.metadata['album']}" for d in context_docs])
    
    # BERTScore
    bert_scores = calculate_bertscore([generated], [reference])
    
    # Faithfulness
    faithfulness = calculate_faithfulness(generated, context_docs)
    
    # Response length
    response_length = len(generated.split())
    
    return {
        'bertscore_f1': bert_scores['f1'],
        'faithfulness': faithfulness,
        'response_length': response_length
    }

print("Generation quality metrics ready!")

## Cell 10: Experiment 1 - Main Retrieval Evaluation

In [ ]:
def run_retrieval_evaluation(test_queries: List[Dict]) -> pd.DataFrame:
    """Run comprehensive retrieval evaluation."""
    results = []
    
    for test in tqdm(test_queries, desc="Evaluating retrieval"):
        query = test['query']
        
        # Measure latency
        start = datetime.now()
        docs = retrieve_documents(query)
        latency = (datetime.now() - start).total_seconds()
        
        # Attribute evaluation
        attr_scores = evaluate_attributes(docs, test)
        
        # Diversity
        diversity = calculate_genre_diversity(docs)
        
        # Results
        result = {
            'query': query,
            'type': test['type'],
            'genre_match': attr_scores['genre_match'],
            'year_match': attr_scores['year_match'],
            'score_match': attr_scores['score_match'],
            'diversity': diversity,
            'latency': latency,
            'num_results': len(docs)
        }
        
        results.append(result)
    
    return pd.DataFrame(results)

# Run evaluation
print("Running main retrieval evaluation...")
retrieval_results = run_retrieval_evaluation(TEST_QUERIES)

# Summary by query type
print("\n" + "="*60)
print("RETRIEVAL EVALUATION RESULTS")
print("="*60)

summary = retrieval_results.groupby('type').agg({
    'genre_match': 'mean',
    'year_match': 'mean',
    'score_match': 'mean',
    'diversity': 'mean',
    'latency': 'mean'
}).round(3)

print("\nResults by Query Type:")
print(summary.to_string())

# Overall
print("\n" + "-"*60)
print("Overall Results:")
print(f"  Avg Genre Match: {retrieval_results['genre_match'].mean():.3f}")
print(f"  Avg Year Match: {retrieval_results['year_match'].mean():.3f}")
print(f"  Avg Score Match: {retrieval_results['score_match'].mean():.3f}")
print(f"  Avg Diversity: {retrieval_results['diversity'].mean():.3f}")
print(f"  Avg Latency: {retrieval_results['latency'].mean():.3f}s")

# Save results
retrieval_results.to_csv(f"{config.RESULTS_DIR}/retrieval_evaluation.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/retrieval_evaluation.csv")

## Cell 11: Experiment 2 - Ablation Study

In [ ]:
def run_ablation_study(test_queries: List[Dict]) -> pd.DataFrame:
    """Run ablation study with different configurations."""
    
    configs = [
        {'name': 'Full System', 'expansion': True, 'adaptive': True, 'reranking': True},
        {'name': 'No Query Expansion', 'expansion': False, 'adaptive': True, 'reranking': True},
        {'name': 'No Adaptive Weights', 'expansion': True, 'adaptive': False, 'reranking': True},
        {'name': 'No Reranking', 'expansion': True, 'adaptive': True, 'reranking': False},
        {'name': 'Vector Only', 'expansion': False, 'adaptive': False, 'reranking': False, 'retriever': 'vector'},
        {'name': 'BM25 Only', 'expansion': False, 'adaptive': False, 'reranking': False, 'retriever': 'bm25'},
        {'name': 'Ensemble Only', 'expansion': False, 'adaptive': False, 'reranking': False}
    ]
    
    results = []
    
    for cfg in tqdm(configs, desc="Ablation configurations"):
        genre_matches = []
        year_matches = []
        diversities = []
        latencies = []
        
        for test in test_queries:
            query = test['query']
            start = datetime.now()
            
            # Get documents based on config
            if 'retriever' in cfg:
                search_query = expand_query(query) if cfg.get('expansion') else query
                docs = retrievers[cfg['retriever']].invoke(search_query)[:config.RERANK_TOP_N]
            else:
                docs = retrieve_documents(
                    query,
                    use_expansion=cfg['expansion'],
                    use_adaptive=cfg['adaptive'],
                    use_reranking=cfg['reranking']
                )
            
            latency = (datetime.now() - start).total_seconds()
            
            # Evaluate
            attr = evaluate_attributes(docs, test)
            genre_matches.append(attr['genre_match'])
            year_matches.append(attr['year_match'])
            diversities.append(calculate_genre_diversity(docs))
            latencies.append(latency)
        
        results.append({
            'Configuration': cfg['name'],
            'Genre Match': np.mean(genre_matches),
            'Year Match': np.mean(year_matches),
            'Diversity': np.mean(diversities),
            'Latency (s)': np.mean(latencies)
        })
    
    return pd.DataFrame(results)

# Run ablation
print("Running ablation study...")
ablation_results = run_ablation_study(TEST_QUERIES)

print("\n" + "="*60)
print("ABLATION STUDY RESULTS")
print("="*60)
print(ablation_results.to_string(index=False))

# Calculate improvements
full_genre = ablation_results[ablation_results['Configuration'] == 'Full System']['Genre Match'].values[0]
print("\n" + "-"*60)
print("Improvement Analysis:")
for _, row in ablation_results.iterrows():
    if row['Configuration'] != 'Full System':
        diff = full_genre - row['Genre Match']
        pct = (diff / row['Genre Match'] * 100) if row['Genre Match'] > 0 else 0
        print(f"  vs {row['Configuration']}: +{diff:.3f} ({pct:.1f}%)")

# Save
ablation_results.to_csv(f"{config.RESULTS_DIR}/ablation_study.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/ablation_study.csv")

## Cell 12: Experiment 3 - Multi-LLM Comparison

In [ ]:
def create_rag_chain(llm):
    """Create RAG chain for evaluation."""
    qa_template = """You are a music recommendation assistant. Based on the album reviews below, recommend albums that match the user's query.

ALBUM REVIEWS:
{context}

QUERY: {question}

Recommend 3-5 albums with brief explanations. Include artist, album, year, and score."""
    
    # Custom retriever wrapper
    class CustomRetriever:
        def invoke(self, query):
            return retrieve_documents(query)
        def get_relevant_documents(self, query):
            return self.invoke(query)
    
    from langchain.chains import RetrievalQA
    
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=CustomRetriever(),
        return_source_documents=True,
        chain_type_kwargs={"prompt": PromptTemplate(template=qa_template, input_variables=["context", "question"])}
    )
    return chain

def evaluate_llm(model_name: str, test_queries: List[Dict], n_samples: int = 20) -> Dict:
    """Evaluate a single LLM."""
    print(f"\nEvaluating {model_name}...")
    
    # Load LLM
    llm, tokenizer = load_llm(model_name)
    chain = create_rag_chain(llm)
    
    # Sample queries for efficiency
    sample_queries = test_queries[:n_samples]
    
    results = {
        'bertscore_f1': [],
        'faithfulness': [],
        'response_length': [],
        'latency': []
    }
    
    for test in tqdm(sample_queries, desc=f"Testing {model_name}"):
        query = test['query']
        
        try:
            start = datetime.now()
            response = chain.invoke({"query": query})
            latency = (datetime.now() - start).total_seconds()
            
            generated = response['result']
            source_docs = response.get('source_documents', [])
            
            # Evaluate generation
            gen_quality = evaluate_generation_quality(query, generated, source_docs)
            
            results['bertscore_f1'].append(gen_quality['bertscore_f1'])
            results['faithfulness'].append(gen_quality['faithfulness'])
            results['response_length'].append(gen_quality['response_length'])
            results['latency'].append(latency)
            
        except Exception as e:
            logger.error(f"Error with {model_name} on query '{query}': {e}")
            continue
    
    # Unload LLM
    del llm, chain
    unload_llm()
    
    return {
        'model': model_name,
        'bertscore_f1': np.mean(results['bertscore_f1']),
        'faithfulness': np.mean(results['faithfulness']),
        'response_length': np.mean(results['response_length']),
        'latency': np.mean(results['latency'])
    }

# Run LLM comparison
print("Running multi-LLM comparison...")
print("Note: This will load each LLM sequentially to manage GPU memory.")

llm_results = []
for model_name in LLM_CONFIGS.keys():
    try:
        result = evaluate_llm(model_name, TEST_QUERIES, n_samples=20)
        llm_results.append(result)
    except Exception as e:
        print(f"Failed to evaluate {model_name}: {e}")
        continue

llm_comparison = pd.DataFrame(llm_results)

print("\n" + "="*60)
print("LLM COMPARISON RESULTS")
print("="*60)
print(llm_comparison.to_string(index=False))

# Save
llm_comparison.to_csv(f"{config.RESULTS_DIR}/llm_comparison.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/llm_comparison.csv")

## Cell 13: Experiment 4 - Diversity Analysis

In [ ]:
def analyze_diversity(test_queries: List[Dict]) -> Dict:
    """Comprehensive diversity analysis."""
    
    results = {
        'genre_entropy': [],
        'artist_coverage': [],
        'year_spread': [],
        'score_spread': []
    }
    
    all_artists = set()
    all_albums = set()
    
    for test in tqdm(test_queries, desc="Analyzing diversity"):
        docs = retrieve_documents(test['query'])
        
        if not docs:
            continue
        
        # Genre entropy
        results['genre_entropy'].append(calculate_genre_diversity(docs))
        
        # Artist coverage (unique artists)
        artists = set(doc.metadata['artist'] for doc in docs)
        results['artist_coverage'].append(len(artists) / len(docs))
        all_artists.update(artists)
        
        # Album coverage
        albums = set(doc.metadata['album'] for doc in docs)
        all_albums.update(albums)
        
        # Year spread (std dev)
        years = [doc.metadata['year'] for doc in docs]
        results['year_spread'].append(np.std(years) if len(years) > 1 else 0)
        
        # Score spread
        scores = [doc.metadata['score'] for doc in docs]
        results['score_spread'].append(np.std(scores) if len(scores) > 1 else 0)
    
    return {
        'avg_genre_entropy': np.mean(results['genre_entropy']),
        'avg_artist_coverage': np.mean(results['artist_coverage']),
        'avg_year_spread': np.mean(results['year_spread']),
        'avg_score_spread': np.mean(results['score_spread']),
        'total_unique_artists': len(all_artists),
        'total_unique_albums': len(all_albums),
        'catalog_coverage': len(all_albums) / len(df) * 100
    }

# Run diversity analysis
print("Running diversity analysis...")
diversity_results = analyze_diversity(TEST_QUERIES)

print("\n" + "="*60)
print("DIVERSITY ANALYSIS RESULTS")
print("="*60)
for key, value in diversity_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.3f}")
    else:
        print(f"  {key}: {value}")

# Save
pd.DataFrame([diversity_results]).to_csv(f"{config.RESULTS_DIR}/diversity_analysis.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/diversity_analysis.csv")

## Cell 14: Experiment 5 - Statistical Significance

In [ ]:
def calculate_significance(test_queries: List[Dict]) -> Dict:
    """Calculate statistical significance with bootstrap CI."""
    
    # Collect scores for full system vs baselines
    full_scores = []
    no_rerank_scores = []
    no_adaptive_scores = []
    
    for test in tqdm(test_queries, desc="Collecting scores"):
        query = test['query']
        
        # Full system
        docs = retrieve_documents(query, use_expansion=True, use_adaptive=True, use_reranking=True)
        full_scores.append(evaluate_attributes(docs, test)['genre_match'])
        
        # No reranking
        docs = retrieve_documents(query, use_expansion=True, use_adaptive=True, use_reranking=False)
        no_rerank_scores.append(evaluate_attributes(docs, test)['genre_match'])
        
        # No adaptive
        docs = retrieve_documents(query, use_expansion=True, use_adaptive=False, use_reranking=True)
        no_adaptive_scores.append(evaluate_attributes(docs, test)['genre_match'])
    
    # Bootstrap CIs
    full_ci = bootstrap_confidence_interval(full_scores, config.CONFIDENCE_LEVEL, config.BOOTSTRAP_SAMPLES)
    no_rerank_ci = bootstrap_confidence_interval(no_rerank_scores, config.CONFIDENCE_LEVEL, config.BOOTSTRAP_SAMPLES)
    no_adaptive_ci = bootstrap_confidence_interval(no_adaptive_scores, config.CONFIDENCE_LEVEL, config.BOOTSTRAP_SAMPLES)
    
    # Paired t-tests
    t_rerank, p_rerank = stats.ttest_rel(full_scores, no_rerank_scores)
    t_adaptive, p_adaptive = stats.ttest_rel(full_scores, no_adaptive_scores)
    
    return {
        'full_system': {
            'mean': np.mean(full_scores),
            'ci_lower': full_ci[0],
            'ci_upper': full_ci[1]
        },
        'no_reranking': {
            'mean': np.mean(no_rerank_scores),
            'ci_lower': no_rerank_ci[0],
            'ci_upper': no_rerank_ci[1],
            'p_value': p_rerank
        },
        'no_adaptive': {
            'mean': np.mean(no_adaptive_scores),
            'ci_lower': no_adaptive_ci[0],
            'ci_upper': no_adaptive_ci[1],
            'p_value': p_adaptive
        }
    }

# Run significance tests
print("Calculating statistical significance...")
significance_results = calculate_significance(TEST_QUERIES)

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE RESULTS")
print("="*60)

for name, results in significance_results.items():
    print(f"\n{name}:")
    print(f"  Mean: {results['mean']:.3f}")
    print(f"  95% CI: [{results['ci_lower']:.3f}, {results['ci_upper']:.3f}]")
    if 'p_value' in results:
        sig = "***" if results['p_value'] < 0.001 else "**" if results['p_value'] < 0.01 else "*" if results['p_value'] < 0.05 else ""
        print(f"  p-value: {results['p_value']:.4f} {sig}")

# Save
sig_df = pd.DataFrame([
    {'config': k, **v} for k, v in significance_results.items()
])
sig_df.to_csv(f"{config.RESULTS_DIR}/statistical_significance.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/statistical_significance.csv")

## Cell 15: Experiment 6 - Efficiency Analysis

In [ ]:
def analyze_efficiency() -> Dict:
    """Analyze system efficiency metrics."""
    
    # Memory usage
    process = psutil.Process()
    cpu_memory = process.memory_info().rss / (1024 ** 3)
    
    gpu_memory = 0
    if torch.cuda.is_available():
        gpu_memory = torch.cuda.memory_allocated() / (1024 ** 3)
    
    # Index size
    index_size = 0
    if os.path.exists(config.VECTOR_STORE_PATH):
        for f in os.listdir(config.VECTOR_STORE_PATH):
            index_size += os.path.getsize(os.path.join(config.VECTOR_STORE_PATH, f))
    index_size_mb = index_size / (1024 ** 2)
    
    # Component latency breakdown
    sample_query = "dreamy shoegaze albums"
    
    # Embedding
    start = datetime.now()
    _ = embeddings.embed_query(sample_query)
    embed_latency = (datetime.now() - start).total_seconds()
    
    # Vector retrieval
    start = datetime.now()
    _ = retrievers['vector'].invoke(sample_query)
    vector_latency = (datetime.now() - start).total_seconds()
    
    # BM25 retrieval
    start = datetime.now()
    _ = retrievers['bm25'].invoke(sample_query)
    bm25_latency = (datetime.now() - start).total_seconds()
    
    # Reranking
    docs = retrievers['ensemble'].invoke(sample_query)
    start = datetime.now()
    intent = detect_intent(sample_query)
    _ = rerank_documents(sample_query, docs, intent)
    rerank_latency = (datetime.now() - start).total_seconds()
    
    return {
        'cpu_memory_gb': cpu_memory,
        'gpu_memory_gb': gpu_memory,
        'index_size_mb': index_size_mb,
        'embed_latency_s': embed_latency,
        'vector_retrieval_s': vector_latency,
        'bm25_retrieval_s': bm25_latency,
        'reranking_s': rerank_latency,
        'total_retrieval_s': embed_latency + vector_latency + bm25_latency + rerank_latency
    }

# Run efficiency analysis
print("Analyzing system efficiency...")
efficiency_results = analyze_efficiency()

print("\n" + "="*60)
print("EFFICIENCY ANALYSIS RESULTS")
print("="*60)

print("\nMemory Usage:")
print(f"  CPU Memory: {efficiency_results['cpu_memory_gb']:.2f} GB")
print(f"  GPU Memory: {efficiency_results['gpu_memory_gb']:.2f} GB")
print(f"  Index Size: {efficiency_results['index_size_mb']:.1f} MB")

print("\nLatency Breakdown:")
print(f"  Embedding: {efficiency_results['embed_latency_s']*1000:.1f} ms")
print(f"  Vector Retrieval: {efficiency_results['vector_retrieval_s']*1000:.1f} ms")
print(f"  BM25 Retrieval: {efficiency_results['bm25_retrieval_s']*1000:.1f} ms")
print(f"  Reranking: {efficiency_results['reranking_s']*1000:.1f} ms")
print(f"  Total Retrieval: {efficiency_results['total_retrieval_s']*1000:.1f} ms")

# Save
pd.DataFrame([efficiency_results]).to_csv(f"{config.RESULTS_DIR}/efficiency_analysis.csv", index=False)
print(f"\nResults saved to {config.RESULTS_DIR}/efficiency_analysis.csv")

## Cell 16: Generate All Result Tables

In [ ]:
print("="*60)
print("SUMMARY OF ALL EXPERIMENTS")
print("="*60)

# Table 1: Main Results
print("\n### Table 1: Main Retrieval Results")
main_summary = retrieval_results.groupby('type').agg({
    'genre_match': 'mean',
    'year_match': 'mean',
    'diversity': 'mean',
    'latency': 'mean'
}).round(3)
print(tabulate(main_summary.reset_index(), headers='keys', tablefmt='pipe', showindex=False))

# Table 2: Ablation
print("\n### Table 2: Ablation Study")
print(tabulate(ablation_results, headers='keys', tablefmt='pipe', showindex=False))

# Table 3: LLM Comparison
if len(llm_results) > 0:
    print("\n### Table 3: LLM Comparison")
    print(tabulate(llm_comparison, headers='keys', tablefmt='pipe', showindex=False))

# Table 4: Diversity
print("\n### Table 4: Diversity Metrics")
div_table = pd.DataFrame([{
    'Metric': 'Genre Entropy', 'Value': f"{diversity_results['avg_genre_entropy']:.3f}"
}, {
    'Metric': 'Artist Coverage', 'Value': f"{diversity_results['avg_artist_coverage']:.3f}"
}, {
    'Metric': 'Catalog Coverage', 'Value': f"{diversity_results['catalog_coverage']:.1f}%"
}])
print(tabulate(div_table, headers='keys', tablefmt='pipe', showindex=False))

# Table 5: Efficiency
print("\n### Table 5: Efficiency Metrics")
eff_table = pd.DataFrame([{
    'Component': 'Total Retrieval',
    'Latency (ms)': f"{efficiency_results['total_retrieval_s']*1000:.1f}",
    'Memory (GB)': f"{efficiency_results['cpu_memory_gb']:.2f}"
}])
print(tabulate(eff_table, headers='keys', tablefmt='pipe', showindex=False))

print("\n" + "="*60)
print(f"All results saved to {config.RESULTS_DIR}/")
print("="*60)

## Cell 17: Visualizations

In [ ]:
# Create publication-ready figures
fig = plt.figure(figsize=(16, 12))

# 1. Ablation study bar chart
ax1 = fig.add_subplot(2, 2, 1)
configs = ablation_results['Configuration']
scores = ablation_results['Genre Match']
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(configs))]
bars = ax1.barh(configs, scores, color=colors)
ax1.set_xlabel('Genre Match Score')
ax1.set_title('Ablation Study Results')
ax1.axvline(x=scores[0], color='red', linestyle='--', alpha=0.5)

# 2. Results by query type
ax2 = fig.add_subplot(2, 2, 2)
type_results = retrieval_results.groupby('type')['genre_match'].mean().sort_values()
type_results.plot(kind='barh', ax=ax2, color='#9b59b6')
ax2.set_xlabel('Genre Match Score')
ax2.set_title('Performance by Query Type')

# 3. Latency breakdown
ax3 = fig.add_subplot(2, 2, 3)
latency_data = {
    'Embedding': efficiency_results['embed_latency_s'] * 1000,
    'Vector': efficiency_results['vector_retrieval_s'] * 1000,
    'BM25': efficiency_results['bm25_retrieval_s'] * 1000,
    'Reranking': efficiency_results['reranking_s'] * 1000
}
ax3.pie(latency_data.values(), labels=latency_data.keys(), autopct='%1.1f%%', colors=plt.cm.Set3.colors)
ax3.set_title('Latency Breakdown')

# 4. Diversity metrics
ax4 = fig.add_subplot(2, 2, 4)
div_metrics = {
    'Genre\nEntropy': diversity_results['avg_genre_entropy'],
    'Artist\nCoverage': diversity_results['avg_artist_coverage'],
    'Year\nSpread': diversity_results['avg_year_spread'] / 10  # Normalize
}
ax4.bar(div_metrics.keys(), div_metrics.values(), color=['#e74c3c', '#f39c12', '#1abc9c'])
ax4.set_ylabel('Score')
ax4.set_title('Diversity Metrics')
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(f"{config.RESULTS_DIR}/experiment_results.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to {config.RESULTS_DIR}/experiment_results.png")

## Cell 18: Export to LaTeX

In [ ]:
# Generate LaTeX tables for paper

def to_latex_table(df, caption, label):
    """Convert DataFrame to LaTeX table."""
    latex = df.to_latex(index=False, float_format="%.3f")
    latex = latex.replace('\\begin{tabular}', f'\\begin{{table}}[h]\n\\centering\n\\caption{{{caption}}}\n\\label{{{label}}}\n\\begin{{tabular}}')
    latex = latex.replace('\\end{tabular}', '\\end{tabular}\n\\end{table}')
    return latex

# Main results table
main_latex = to_latex_table(
    main_summary.reset_index().round(3),
    'Main Retrieval Results by Query Type',
    'tab:main_results'
)

# Ablation table
ablation_latex = to_latex_table(
    ablation_results.round(3),
    'Ablation Study Results',
    'tab:ablation'
)

# Save LaTeX
with open(f"{config.RESULTS_DIR}/tables.tex", 'w') as f:
    f.write("% Main Results\n")
    f.write(main_latex)
    f.write("\n\n% Ablation Study\n")
    f.write(ablation_latex)

print("LaTeX tables saved to experiment_results/tables.tex")

print("\n" + "="*60)
print("ALL EXPERIMENTS COMPLETE!")
print("="*60)
print(f"\nResults directory: {config.RESULTS_DIR}/")
print("Files generated:")
for f in os.listdir(config.RESULTS_DIR):
    size = os.path.getsize(os.path.join(config.RESULTS_DIR, f)) / 1024
    print(f"  - {f} ({size:.1f} KB)")